# 08A — Fireline characteristics and condition–outcome analysis

**New 08 series: learning which recorded conditions accompany Held and Burned Over outcomes.**

Run this notebook top to bottom in Antigravity using the project's **.venv / Python 3** kernel. All analysis code is contained here. No network downloads, package installations, model fitting, or changes to the 05/06-series models are performed.

### What 08A answers
1. How much reported width, fireline type, and source-record length information is available, and which values need review?
2. How do recorded outcomes vary across transparent weather, terrain, canopy, approach, and fireline-characteristic groups?
3. Do group differences also appear when comparing portions of the **same fire**?

### What the outputs mean
- The authority for inclusion is the existing **304,143 fireline units / 381 fires** master. Labels, prediction times, and membership remain frozen.
- Width is **operations-reported**, not independently verified physical width. Positive values are not automatically reliable. Missing widths are never guessed.
- Source-record length and the analytical fireline-unit length are **not necessarily the entire constructed fireline length**. The 250 m cap is an analysis design choice.
- HRRR relative humidity describes atmospheric humidity, **not fuel moisture**. Future valid hours refer to forecasts selected by the original pipeline.
- Bins below are transparent analyst choices made **after earlier exploration of this archive**. They are not preregistered, optimized thresholds, causal effects, or operating rules.
- Year and saved-fold slices in 08A are **descriptive**, not new held-out prediction tests. Saved 05I regime membership also used all years during final rule construction.
- Fire-average rates deliberately give every represented fire equal weight; pooled rates answer a different question. A fire can occur in several groups.

**Planned next stage (not implemented here):** 08B will develop a small interpretable condition model using development fires only, freeze its rules, and evaluate recorded outcomes on excluded fires and forward years. Width/length inclusion depends on this audit. Independent confirmation still needs additional fires because this archive has already informed development.


## 0. Configuration

The default bins emphasize readable contrasts, not the previously learned 05I cut points. Do not tune bins until a desired result appears. Changing a bin creates another exploratory analysis and a new run.

Every run writes to a **new timestamped folder** under outputs/08_condition_outcomes. Existing datasets and results are never overwritten. Sparse groups remain in the tables but are flagged; the support thresholds below are reporting screens, not statistical guarantees.


In [ ]:
import os  # Configurable project root for the public release.
from pathlib import Path
from datetime import datetime, timezone
import importlib.util
import hashlib
import json
import platform
import sys
import uuid
import zlib

required_packages = ["numpy", "pandas", "pyarrow", "matplotlib", "IPython"]
missing_packages = [name for name in required_packages if importlib.util.find_spec(name) is None]
if missing_packages:
    raise ImportError(
        "Use the project's .venv kernel. Missing packages: " + ", ".join(missing_packages)
        + ". This notebook does not install packages automatically."
    )

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display, Markdown

def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents,
                  Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))]
    for candidate in candidates:
        if (candidate / "data/processed/modeling_table_assembly_v1/"
                        "fireline_hold_modeling_master_v1.parquet").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Open Antigravity in the project folder or set PROJECT_ROOT manually.")

PROJECT_ROOT = find_project_root()
BUILD_ID = "08A_V1_FIRELINE_CHARACTERISTICS_AND_CONDITION_OUTCOMES"
EXPECTED_UNITS = 304_143
EXPECTED_FIRES = 381
RANDOM_SEED = 20260930
BOOTSTRAP_REPLICATES = 2_000
MIN_GROUP_UNITS = 200
MIN_GROUP_FIRES = 20
MIN_OUTCOME_COUNT = 20
MIN_UNITS_PER_FIRE_SIDE = 5

# Review flag only: NOT a maximum feasible width or a validated quality boundary.
# All reported widths and flags remain in the saved table. Screened-width
# outcome tables omit nonpositive/nonfinite values and flag values above this limit.
WIDTH_REVIEW_THRESHOLD_FT = 300.0
WIDTH_REVIEW_SENSITIVITY_FT = [100.0, 300.0, 1000.0]

MASTER_PATH = PROJECT_ROOT / "data/processed/modeling_table_assembly_v1/fireline_hold_modeling_master_v1.parquet"
UNIT_METADATA_PATH = PROJECT_ROOT / "data/processed/fireline_corrected_local_250m_10m/frozen_cohort/fireline_ml_dynamic_timing_cohort_frozen_v1.parquet"
SOURCE_PATH = PROJECT_ROOT / "data/processed/fireline/fireline_qaqc_version_resolved.parquet"
FOLD_PATH = PROJECT_ROOT / "data/processed/modeling_design_v1/unit_outer_fold_and_training_weights_v1.parquet"
LEGACY_ASSIGNMENT_PATH = PROJECT_ROOT / "data/processed/input_regimes_v1/FINAL_INPUT_REGIME_ASSIGNMENTS_V1.parquet"
LEGACY_RULE_PATH = PROJECT_ROOT / "outputs/input_regimes_v1/PHYSICAL_INPUT_REGIMES_V1.csv"

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + uuid.uuid4().hex[:6]
RUN_DIR = PROJECT_ROOT / "outputs/08_condition_outcomes" / ("08A_" + RUN_ID)
FIGURE_DIR = RUN_DIR / "figures"
IS_SMOKE_TEST = False

# (column, display label, units, family, bin edges, bin labels, valid min/max).
# All numeric bins are left-closed, right-open. Physical bounds are handled
# separately, so RH=100% remains a valid observation in the last numeric bin.
SPECS = {
    "humidity": {
        "column": "hrrr_window_0_18h_rh2_pct_mean",
        "label": "Mean forecast relative humidity, 0–18 h", "units": "%",
        "family": "HRRR forecast weather",
        "edges": [0, 20, 40, 60, 80, np.inf],
        "labels": ["0–<20", "20–<40", "40–<60", "60–<80", "80–100"],
        "bounds": [0, 100]},
    "gust": {
        "column": "hrrr_gust_mps_p18h",
        "label": "Forecast gust at +18 h", "units": "m/s",
        "family": "HRRR forecast weather",
        "edges": [0, 5, 10, 15, 20, np.inf],
        "labels": ["0–<5", "5–<10", "10–<15", "15–<20", "20+"],
        "bounds": [0, None]},
    "wind": {
        "column": "hrrr_window_0_18h_wind10_speed_mean_mps",
        "label": "Mean forecast 10 m wind speed, 0–18 h", "units": "m/s",
        "family": "HRRR forecast weather",
        "edges": [0, 3, 6, 9, 12, np.inf],
        "labels": ["0–<3", "3–<6", "6–<9", "9–<12", "12+"],
        "bounds": [0, None]},
    "temperature": {
        "column": "hrrr_window_0_18h_t2_c_max",
        "label": "Maximum forecast temperature, 0–18 h", "units": "degrees C",
        "family": "HRRR forecast weather",
        "edges": [-np.inf, 10, 20, 30, 40, np.inf],
        "labels": ["<10", "10–<20", "20–<30", "30–<40", "40+"],
        "bounds": [None, None]},
    "slope": {
        "column": "terrain_slope_mean_deg",
        "label": "Mean sampled terrain slope", "units": "degrees",
        "family": "USGS 3DEP terrain",
        "edges": [0, 10, 20, 30, 40, np.inf],
        "labels": ["0–<10", "10–<20", "20–<30", "30–<40", "40–90"],
        "bounds": [0, 90]},
    "elevation": {
        "column": "terrain_elevation_mean_m",
        "label": "Mean sampled elevation", "units": "m",
        "family": "USGS 3DEP terrain",
        "edges": [-np.inf, 500, 1000, 1500, 2000, np.inf],
        "labels": ["<500", "500–<1000", "1000–<1500", "1500–<2000", "2000+"],
        "bounds": [None, None]},
    "crown": {
        "column": "landfire_protected_far_crown_depth_p90",
        "label": "Opposite-side crown-depth 90th percentile, 150–300 m offsets",
        "units": "m", "family": "LANDFIRE canopy",
        "edges": [0, 5, 10, 20, 30, np.inf],
        "labels": ["0–<5", "5–<10", "10–<20", "20–<30", "30+"],
        "bounds": [0, None]},
    "distance": {
        "column": "local_fire_to_line_distance_geom_m",
        "label": "Mapped fire-perimeter-to-fireline distance", "units": "m",
        "family": "NIFC / NIROPS fire approach",
        "edges": [0, 25, 50, 100, 250, 1000, np.inf],
        "labels": ["0–<25", "25–<50", "50–<100", "100–<250", "250–<1000", "1000+"],
        "bounds": [0, None]},
    "width_screened": {
        "column": "reported_width_m_screened",
        "label": "Reported width at or below the review flag (not verified)", "units": "m",
        "family": "NIFC operations-reported attribute",
        "edges": [0, 3, 6, 12, 30, np.inf],
        "labels": [">0–<3", "3–<6", "6–<12", "12–<30", "30+"],
        "bounds": [0, None]},
    "source_length": {
        "column": "source_geometry_length_m",
        "label": "Mapped source-record length (not whole constructed fireline)", "units": "m",
        "family": "NIFC source geometry",
        "edges": [0, 50, 100, 250, 500, 1000, np.inf],
        "labels": [">0–<50", "50–<100", "100–<250", "250–<500", "500–<1000", "1000+"],
        "bounds": [0, None]},
    "analysis_length": {
        "column": "actual_line_length_m",
        "label": "Analytical fireline-unit length (design diagnostic)", "units": "m",
        "family": "Analysis segmentation",
        "edges": [0, 25, 50, 100, 200, np.inf],
        "labels": [">0–<25", "25–<50", "50–<100", "100–<200", "200–250"],
        "bounds": [0, 250.000001]},
}
PAIR_SPECS = [
    ("humidity", "gust"),
    ("humidity", "crown"),
    ("fireline_type", "humidity"),
    ("width_screened", "gust"),
]

pd.set_option("display.max_columns", 25)
pd.set_option("display.max_rows", 25)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 220,
                     "axes.spines.top": False, "axes.spines.right": False})
print(BUILD_ID)
print("Project:", PROJECT_ROOT)
print("Planned output:", RUN_DIR)
print("No models will be trained. Bootstrap replicates:", BOOTSTRAP_REPLICATES)


## 1. Reproducibility and helper functions

Bootstrap intervals resample **complete fires**, with the bins held fixed. They describe variation among observed fires, not uncertainty from selecting bins, changing eligibility, or retraining models. No multiple-comparison correction or significance-based rule selection is performed.

The same fire-average estimand is used throughout: calculate each fire's Held fraction in a group, then average the fire fractions equally. Pooled rates are also supplied. Counts of fires across groups must not be added together.


In [ ]:
def require(condition, message):
    if not bool(condition):
        raise AssertionError(message)

def read_columns(path, columns):
    if not path.is_file():
        raise FileNotFoundError(path)
    available = pq.ParquetFile(path).schema_arrow.names
    missing = sorted(set(columns) - set(available))
    require(not missing, f"{path.name}: missing columns {missing}")
    return pd.read_parquet(path, columns=list(dict.fromkeys(columns)), engine="pyarrow")

def check_unique(frame, columns, description):
    require(frame[columns].notna().all().all(), f"{description}: missing key")
    require(not frame.duplicated(columns).any(), f"{description}: duplicate key {columns}")

def identity_digest(frame):
    identity = frame[["unit_uid", "_irwin_norm", "Year", "hold_label"]].sort_values("unit_uid")
    payload = pd.util.hash_pandas_object(identity, index=False).to_numpy().tobytes()
    return hashlib.sha256(payload).hexdigest()

def json_safe(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        if np.isposinf(value): return "+inf"
        if np.isneginf(value): return "-inf"
        if np.isnan(value): return None
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    return value

def write_json(name, value):
    (RUN_DIR / name).write_text(json.dumps(json_safe(value), indent=2, allow_nan=False), encoding="utf-8")

def seed_for(key):
    return (RANDOM_SEED + zlib.crc32(str(key).encode("utf-8"))) % (2**32)

def bootstrap_mean(values, key):
    values = np.asarray(values, dtype=float)
    require(np.isfinite(values).all(), "Bootstrap received nonfinite values")
    if len(values) < 2:
        return np.nan, np.nan
    rng = np.random.default_rng(seed_for(key))
    draws = np.empty(BOOTSTRAP_REPLICATES)
    for start in range(0, BOOTSTRAP_REPLICATES, 128):
        stop = min(start + 128, BOOTSTRAP_REPLICATES)
        indices = rng.integers(0, len(values), size=(stop-start, len(values)))
        draws[start:stop] = values[indices].mean(axis=1)
    return tuple(np.quantile(draws, [0.025, 0.975]))

def numeric_bins(series, spec):
    x = pd.to_numeric(series, errors="coerce")
    values = x.to_numpy(dtype=float)
    valid = pd.Series(np.isfinite(values), index=series.index)
    lo, hi = spec["bounds"]
    if lo is not None: valid &= x.ge(lo)
    if hi is not None: valid &= x.le(hi)
    out = pd.Series("Missing", index=series.index, dtype="string")
    out.loc[series.notna() & x.isna()] = "Non-numeric"
    out.loc[x.notna() & ~valid] = "Invalid / nonfinite"
    binned = pd.cut(x.where(valid), bins=spec["edges"], labels=spec["labels"],
                    right=False, include_lowest=True)
    out.loc[valid] = binned.loc[valid].astype("string")
    require(out.notna().all(), f"Bin edges do not cover {spec['label']}")
    return out

def summarize_groups(frame, groups, context, with_intervals=True):
    rows = []
    # Explicit list avoids a scalar/tuple ambiguity between one and several keys.
    grouper = groups[0] if len(groups) == 1 else groups
    for keys, part in frame.groupby(grouper, observed=True, dropna=False, sort=True):
        keys = (keys,) if len(groups) == 1 else keys
        per_fire = part.groupby("_irwin_norm")["hold_label"].agg(["size", "sum", "mean"])
        n = len(part)
        held = int(part["hold_label"].sum())
        rates = per_fire["mean"].to_numpy(dtype=float)
        lower, upper = bootstrap_mean(rates, (context, keys)) if with_intervals else (np.nan, np.nan)
        enough = (n >= MIN_GROUP_UNITS and len(per_fire) >= MIN_GROUP_FIRES
                  and min(held, n-held) >= MIN_OUTCOME_COUNT)
        row = dict(zip(groups, keys))
        row.update({
            "analysis": context,
            "n_fireline_units": n, "n_source_records": part["source_record_key"].nunique(),
            "n_fires": len(per_fire), "n_held": held, "n_burned_over": n-held,
            "pooled_held_rate": held/n, "pooled_burned_over_rate": 1-held/n,
            "fire_average_held_rate": rates.mean(),
            "fire_average_burned_over_rate": 1-rates.mean(),
            "held_ci_lower": lower, "held_ci_upper": upper,
            "burned_over_ci_lower": 1-upper, "burned_over_ci_upper": 1-lower,
            "n_fires_with_both_outcomes": int(((per_fire["sum"] > 0) &
                                             (per_fire["sum"] < per_fire["size"])).sum()),
            "median_fireline_units_per_fire": float(per_fire["size"].median()),
            "meets_reporting_screen": enough,
            "degenerate_fire_bootstrap": bool(len(rates) > 1 and np.ptp(rates) == 0),
            "evidence_status": "exploratory; fixed groups; no independent validation",
        })
        rows.append(row)
    return pd.DataFrame(rows)

def numeric_summary(series):
    x = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return dict(n_finite=0, mean=np.nan, median=np.nan, p05=np.nan,
                    p25=np.nan, p75=np.nan, p95=np.nan, minimum=np.nan, maximum=np.nan)
    q = np.quantile(x, [0.05, 0.25, 0.5, 0.75, 0.95])
    return dict(n_finite=len(x), mean=x.mean(), median=q[2], p05=q[0],
                p25=q[1], p75=q[3], p95=q[4], minimum=x.min(), maximum=x.max())

print("Helper functions ready.")


## 2. Recover attributes without changing the study population

The final modeling master determines inclusion. An earlier saved metadata table supplies the source-geometry key and analytical length; it does **not** add its extra fireline units. The source join uses **fire identifier plus geometry hash**. A hash alone is not unique across fires.

Every join must preserve the master identities and outcomes. Missing source records are an error; a matched source record with an unreported width is a valid missing-attribute case. No predictor values or existing predictions are recomputed.


In [ ]:
require(BOOTSTRAP_REPLICATES >= 20, "Use at least 20 replicates for a smoke test; 2,000 for analysis.")
require(WIDTH_REVIEW_THRESHOLD_FT > 0, "The review flag must be positive.")
RUN_DIR.mkdir(parents=True, exist_ok=False)
FIGURE_DIR.mkdir()
input_paths = [MASTER_PATH, UNIT_METADATA_PATH, SOURCE_PATH, FOLD_PATH]
input_paths += [p for p in [LEGACY_ASSIGNMENT_PATH, LEGACY_RULE_PATH] if p.is_file()]
input_stats = {
    str(p): {"bytes": p.stat().st_size, "modified_ns": p.stat().st_mtime_ns}
    for p in input_paths
}
source_columns = ["_irwin_norm", "_geom_hash", "_engagement_norm", "Year",
                  "FeatureCategory", "StrategicLineType", "LineWidthFeet",
                  "LengthFeet", "LineLengthGeodesicKM", "_length_km_geom"]
identity_columns = ["unit_uid", "_irwin_norm", "Year", "hold_label", "_engagement_norm"]
physical_columns = [spec["column"] for key, spec in SPECS.items()
                    if key not in ["width_screened", "source_length", "analysis_length"]]
anchor_columns = ["prediction_anchor_utc", "exact_t0_observation_utc"]
master = read_columns(MASTER_PATH, identity_columns + anchor_columns + physical_columns)
require(master[anchor_columns].notna().all().all(), "Missing saved prediction timestamp.")
require(master["prediction_anchor_utc"].equals(master["exact_t0_observation_utc"]),
        "Saved prediction anchor differs from its matched NIROPS observation.")
notebook_path = PROJECT_ROOT / "notebooks/08A_fireline_characteristics_and_condition_outcomes.ipynb"
notebook_source_hash = None
if notebook_path.is_file():
    notebook_document = json.loads(notebook_path.read_text(encoding="utf-8"))
    notebook_sources = [cell["source"] for cell in notebook_document["cells"]]
    notebook_source_hash = hashlib.sha256(
        json.dumps(notebook_sources, ensure_ascii=False).encode("utf-8")).hexdigest()
check_unique(master, ["unit_uid"], "Final master")
require(len(master) == EXPECTED_UNITS, "Unexpected master count: review the intended dataset version.")
require(master["_irwin_norm"].nunique() == EXPECTED_FIRES, "Unexpected number of fires.")
require(master[identity_columns].notna().all().all(), "Missing master identity or label.")
require(master["hold_label"].value_counts().to_dict() == {1: 205926, 0: 98217},
        "Unexpected recorded outcomes.")
require(set(master["Year"].unique()) == set(range(2020, 2025)), "Unexpected study years.")
master_identity = identity_digest(master)
master_values_hash = hashlib.sha256(
    pd.util.hash_pandas_object(master.sort_values("unit_uid"), index=False).to_numpy().tobytes()
).hexdigest()

metadata = read_columns(UNIT_METADATA_PATH, identity_columns +
                        ["_geom_hash", "actual_line_length_m", "FeatureCategory"])
check_unique(metadata, ["unit_uid"], "Saved fireline-unit metadata")
metadata = metadata.rename(columns={k: "unit_meta_" + k
                                   for k in identity_columns if k != "unit_uid"})
df = master.merge(metadata, on="unit_uid", how="left", validate="one_to_one",
                  indicator="_metadata_match")
require(df["_metadata_match"].eq("both").all(), "Some master fireline units lack saved metadata.")
for field in identity_columns:
    if field != "unit_uid":
        require(df[field].astype("string").equals(df["unit_meta_" + field].astype("string")),
                "Master/metadata disagreement: " + field)
df = df.drop(columns=["_metadata_match"] + ["unit_meta_" + k for k in identity_columns if k != "unit_uid"])
require(df["actual_line_length_m"].gt(0).all() and
        df["actual_line_length_m"].le(250.000001).all(), "Analytical length is outside (0,250] m.")

source = read_columns(SOURCE_PATH, source_columns)
require(len(source) == 570111, "Unexpected source archive version or count.")
check_unique(source, ["_irwin_norm", "_geom_hash"], "Version-resolved source archive")
source["source_record_key"] = source["_irwin_norm"].astype(str) + "|" + source["_geom_hash"].astype(str)
check_unique(source, ["source_record_key"], "Composite parent key")
source_for_join = source.rename(columns={
    "_engagement_norm": "source_outcome", "Year": "source_year",
    "FeatureCategory": "source_feature_category"})
df = df.merge(source_for_join, on=["_irwin_norm", "_geom_hash"], how="left",
              validate="many_to_one", indicator="_source_match")
require(df["_source_match"].eq("both").all(), "Missing source record: stop rather than count as missing width.")
for left, right in [("_engagement_norm", "source_outcome"),
                    ("Year", "source_year"), ("FeatureCategory", "source_feature_category")]:
    require(df[left].astype("string").fillna("<MISSING>").equals(
            df[right].astype("string").fillna("<MISSING>")), f"Source agreement failed: {left}")
df = df.drop(columns="_source_match")

folds = read_columns(FOLD_PATH, ["unit_uid", "outer_fold"])
check_unique(folds, ["unit_uid"], "Saved outer-fold assignments")
df = df.merge(folds, on="unit_uid", how="left", validate="one_to_one")
require(df["outer_fold"].notna().all(), "Missing saved fold.")
require(df.groupby("_irwin_norm")["outer_fold"].nunique().eq(1).all(), "A fire spans saved folds.")
require(set(df["outer_fold"].unique()) == set(range(5)), "Expected five saved folds.")
require(len(df) == len(master) and identity_digest(df) == master_identity, "Joins changed the population.")
require(df["source_record_key"].nunique() == 142999, "Unexpected number of represented source records.")
print(f"Verified: {len(df):,} fireline units / {df['_irwin_norm'].nunique():,} fires / "
      f"{df['source_record_key'].nunique():,} distinct source records.")
print("All source records matched; outcomes and saved fire-fold membership agree.")


## 3. Audit width, length, and type before interpreting outcomes

One source width can be inherited by several fireline units. Coverage is therefore shown for **fireline units, distinct source records, and fires**, not as independent width measurements.

Width categories retain missing, zero, negative, non-numeric, nonfinite, and unusually large values separately. The configurable 300 ft flag is a **manual-review trigger**, not evidence that widths below it are correct or that larger widths are impossible. Raw values remain in the exported table. Sensitivity summaries use other review flags without choosing a "best" flag from the outcomes.

The width outcome analysis is conditional on having a positive reported value at or below the review flag. Its results may not represent the fireline units without reported width. Reported source length, GIS-derived source length, and the ≤250 m analytical length remain separate.


In [ ]:
def width_status(series):
    x = pd.to_numeric(series, errors="coerce")
    finite = pd.Series(np.isfinite(x.to_numpy(dtype=float)), index=series.index)
    status = pd.Series("Missing", index=series.index, dtype="string")
    status.loc[series.notna() & x.isna()] = "Non-numeric"
    status.loc[x.notna() & ~finite] = "Nonfinite"
    status.loc[finite & x.lt(0)] = "Negative"
    status.loc[finite & x.eq(0)] = "Zero"
    status.loc[finite & x.gt(0) & x.le(WIDTH_REVIEW_THRESHOLD_FT)] = "Positive, at or below review flag"
    status.loc[finite & x.gt(WIDTH_REVIEW_THRESHOLD_FT)] = "Positive, above review flag"
    return status

def add_attributes(frame):
    out = frame.copy()
    out["width_status"] = width_status(out["LineWidthFeet"])
    width_ft = pd.to_numeric(out["LineWidthFeet"], errors="coerce")
    out["reported_width_m_raw"] = width_ft * 0.3048
    out["reported_width_m_screened"] = out["reported_width_m_raw"].where(
        out["width_status"].eq("Positive, at or below review flag"))
    out["source_geometry_length_m_raw"] = pd.to_numeric(out["_length_km_geom"], errors="coerce") * 1000
    out["source_geometry_length_m"] = out["source_geometry_length_m_raw"].where(
        np.isfinite(out["source_geometry_length_m_raw"]) & out["source_geometry_length_m_raw"].gt(0))
    out["source_reported_length_m_raw"] = pd.to_numeric(out["LengthFeet"], errors="coerce") * 0.3048
    out["source_geodesic_length_m_raw"] = pd.to_numeric(out["LineLengthGeodesicKM"], errors="coerce") * 1000
    out["fireline_type"] = out["FeatureCategory"].astype("string").fillna("Missing").replace("", "Missing")
    out["strategic_line_type"] = out["StrategicLineType"].astype("string").fillna("Missing").replace("", "Missing")
    return out

df = add_attributes(df)
source_audit = add_attributes(source)
parents = df.drop_duplicates("source_record_key").copy()
require(len(parents) == 142999, "Parent coverage denominator changed.")
tables = {}

def coverage_rows(frame, level):
    rows = []
    for status, part in frame.groupby("width_status", dropna=False, sort=True):
        rows.append({"population": level, "width_status": status,
                     "records": len(part), "population_records": len(frame),
                     "record_fraction": len(part)/len(frame),
                     "n_fires": part["_irwin_norm"].nunique(),
                     "n_source_records": part["source_record_key"].nunique()})
    return rows

tables["attribute_width_coverage"] = pd.DataFrame(
    coverage_rows(source_audit, "Entire version-resolved source archive") +
    coverage_rows(parents, "Distinct source records represented in final dataset") +
    coverage_rows(df, "Final fireline units"))
def width_coverage_by_stratum(frame, stratum):
    rows = []
    for value, population in frame.groupby(stratum, observed=True, dropna=False, sort=True):
        total_units = len(population)
        total_parents = population["source_record_key"].nunique()
        total_fires = population["_irwin_norm"].nunique()
        for status, part in population.groupby("width_status", dropna=False, sort=True):
            rows.append({
                stratum: value, "width_status": status,
                "n_fireline_units": len(part), "stratum_n_fireline_units": total_units,
                "fraction_fireline_units": len(part)/total_units,
                "n_source_records": part["source_record_key"].nunique(),
                "stratum_n_source_records": total_parents,
                "fraction_source_records": part["source_record_key"].nunique()/total_parents,
                "n_fires": part["_irwin_norm"].nunique(), "stratum_n_fires": total_fires,
                # A fire can have several width statuses; these fractions need not sum to one.
                "fraction_fires_with_status": part["_irwin_norm"].nunique()/total_fires,
            })
    result = pd.DataFrame(rows)
    require(np.allclose(result.groupby(stratum)["fraction_fireline_units"].sum(), 1),
            "Width availability fractions do not sum to one within " + stratum)
    return result

tables["width_availability_by_outcome"] = width_coverage_by_stratum(df, "hold_label")
tables["width_availability_by_year"] = width_coverage_by_stratum(df, "Year")
tables["width_availability_by_type"] = width_coverage_by_stratum(df, "fireline_type")

fire_coverage = df.assign(
    width_reported=df["LineWidthFeet"].notna(),
    width_screened=df["reported_width_m_screened"].notna()
).groupby("_irwin_norm").agg(
    n_fireline_units=("unit_uid", "size"), n_source_records=("source_record_key", "nunique"),
    width_reported_fraction=("width_reported", "mean"),
    screened_width_fraction=("width_screened", "mean"),
    recorded_held_fraction=("hold_label", "mean"))
tables["width_coverage_per_fire"] = fire_coverage.reset_index()

sensitivity = []
for flag in WIDTH_REVIEW_SENSITIVITY_FT:
    for frame, level in [(parents, "Distinct represented source records"), (df, "Final fireline units")]:
        x = pd.to_numeric(frame["LineWidthFeet"], errors="coerce")
        eligible = np.isfinite(x) & x.gt(0) & x.le(flag)
        row = {"review_flag_ft": flag, "population": level,
               "n_records_below_flag": int(eligible.sum()), "fraction_below_flag": float(eligible.mean()),
               "n_fires_below_flag": frame.loc[eligible, "_irwin_norm"].nunique(),
               "n_positive_above_flag": int((np.isfinite(x) & x.gt(flag)).sum())}
        if level == "Final fireline units" and eligible.any():
            row["pooled_held_rate_below_flag"] = frame.loc[eligible, "hold_label"].mean()
            row["fire_average_held_rate_below_flag"] = frame.loc[eligible].groupby("_irwin_norm")["hold_label"].mean().mean()
        sensitivity.append(row)
tables["width_review_flag_sensitivity"] = pd.DataFrame(sensitivity)

distribution_rows = []
for name in ["LineWidthFeet", "reported_width_m_screened", "source_reported_length_m_raw",
             "source_geodesic_length_m_raw", "source_geometry_length_m_raw", "actual_line_length_m"] + physical_columns:
    distribution_rows.append({"population": "Final fireline units", "column": name,
                              "n_missing": int(df[name].isna().sum()), **numeric_summary(df[name])})
    if name in ["LineWidthFeet", "source_reported_length_m_raw", "source_geodesic_length_m_raw",
                "source_geometry_length_m_raw"]:
        distribution_rows.append({"population": "Distinct represented source records", "column": name,
                                  "n_missing": int(parents[name].isna().sum()), **numeric_summary(parents[name])})
tables["numeric_attribute_audit"] = pd.DataFrame(distribution_rows)
tables["source_type_coverage"] = summarize_groups(df, ["fireline_type"], "Recorded source fireline type", False)
tables["width_status_outcomes"] = summarize_groups(df, ["width_status"], "Outcome differences by width availability")
parent_attribute_hash = hashlib.sha256(pd.util.hash_pandas_object(
    parents[["source_record_key", "LineWidthFeet", "LengthFeet", "LineLengthGeodesicKM",
             "_length_km_geom", "FeatureCategory", "StrategicLineType"]].sort_values("source_record_key"),
    index=False).to_numpy().tobytes()).hexdigest()
display(tables["attribute_width_coverage"])
display(tables["width_review_flag_sensitivity"])
print("Width is operations-reported and not independently verified. Read coverage before interpreting width groups.")


## 4. Recorded outcomes in transparent condition groups

For every group, the tables show Held and Burned Over counts, fireline-unit/source-record/fire counts, pooled outcome fractions, and equally weighted **fire-average** outcome fractions. The latter first calculates the Held fraction separately in each represented fire, then averages those fractions.

The intervals are 95% percentile intervals from resampling fires. They are not guarantees of prediction accuracy. Missing and flagged observations stay visible as separate rows. Reporting screens flag small groups rather than removing inconvenient results. Year and saved-fold tables describe consistency within this already explored archive; they do not constitute new validation.

Single-variable profiles do not hold other conditions constant. Two-variable tables show combinations, not independently estimated effects.


In [ ]:
for key, spec in SPECS.items():
    df["bin_" + key] = numeric_bins(df[spec["column"]], spec)
# Explain WHY width was not placed in a numeric group, rather than calling all exclusions missing.
width_unavailable = df["reported_width_m_screened"].isna()
df.loc[width_unavailable, "bin_width_screened"] = (
    "Width: " + df.loc[width_unavailable, "width_status"])
length_unavailable = df["source_geometry_length_m"].isna()
df.loc[length_unavailable, "bin_source_length"] = "Missing / nonpositive / nonfinite source length"
df["bin_fireline_type"] = df["fireline_type"]
# Sensitivity profile retains EVERY positive finite reported width, even above the review flag.
positive_width = df["reported_width_m_raw"].where(
    np.isfinite(df["reported_width_m_raw"]) & df["reported_width_m_raw"].gt(0))
df["bin_width_all_positive"] = numeric_bins(positive_width, SPECS["width_screened"])

overall_frame = df.assign(study_group="All retained fireline units")
tables["study_outcome_summary"] = summarize_groups(overall_frame, ["study_group"], "Frozen study population")
profiles, year_profiles, fold_profiles = [], [], []
profile_keys = list(SPECS) + ["fireline_type", "width_all_positive"]
for key in profile_keys:
    column = "bin_" + key
    profile = summarize_groups(df, [column], "Condition profile: " + key)
    require(profile["n_fireline_units"].sum() == EXPECTED_UNITS, "Profile lost observations: " + key)
    profile = profile.rename(columns={column: "condition_group"})
    profile.insert(0, "predictor_key", key)
    profiles.append(profile)
    for slice_column, collection in [("Year", year_profiles), ("outer_fold", fold_profiles)]:
        sliced = summarize_groups(df, [slice_column, column],
                                  f"DESCRIPTIVE {slice_column} slice: {key}", with_intervals=False)
        sliced = sliced.rename(columns={column: "condition_group"})
        sliced.insert(0, "predictor_key", key)
        collection.append(sliced)
tables["condition_profiles"] = pd.concat(profiles, ignore_index=True)
tables["condition_profiles_by_year_descriptive"] = pd.concat(year_profiles, ignore_index=True)
tables["condition_profiles_by_saved_fold_descriptive"] = pd.concat(fold_profiles, ignore_index=True)

pair_tables = []
for first, second in PAIR_SPECS:
    result = summarize_groups(df, ["bin_" + first, "bin_" + second],
                              f"Condition combination: {first} + {second}")
    require(result["n_fireline_units"].sum() == EXPECTED_UNITS, "Combination lost observations.")
    result = result.rename(columns={"bin_" + first: "first_group", "bin_" + second: "second_group"})
    result.insert(0, "first_predictor", first)
    result.insert(1, "second_predictor", second)
    pair_tables.append(result)
tables["condition_combinations"] = pd.concat(pair_tables, ignore_index=True)
tables["recorded_outcomes_by_year"] = summarize_groups(df, ["Year"], "Recorded outcomes by year")
display(tables["study_outcome_summary"])
display(tables["condition_profiles"].query("predictor_key in ['humidity', 'gust']"))
print("All condition groups are descriptive. Missing/flagged groups are retained in CSV outputs.")


## 5. Compare conditions within the same fires

A difference between two groups may arise because the groups contain different fires. Here, each contrast uses only fires containing **both** comparison groups, with at least five fireline units per side. We calculate the difference in Held fractions inside each fire, then average the differences equally across the eligible fires.

The table reports **group B minus group A** in probability units (multiply by 100 for percentage points). Both group rates use exactly the same fires. The bootstrap resamples the paired fire-level differences. No interval is produced with fewer than two fires.

The contrasts below are readable exploratory choices, not optimized thresholds. Even a consistent within-fire difference is not a causal effect: vegetation, suppression activity, and fireline placement can still differ within a fire.


In [ ]:
def paired_fire_contrast(frame, mask_a, mask_b, name, label_a, label_b):
    a = pd.Series(mask_a, index=frame.index).fillna(False).astype(bool)
    b = pd.Series(mask_b, index=frame.index).fillna(False).astype(bool)
    require(not (a & b).any(), "Contrast groups overlap: " + name)
    selected = frame.loc[a | b, ["_irwin_norm", "hold_label", "unit_uid", "source_record_key"]].copy()
    selected["side"] = np.where(a.loc[selected.index], "A", "B")
    g = selected.groupby(["_irwin_norm", "side"])["hold_label"].agg(["size", "mean"])
    counts = g["size"].unstack("side").reindex(columns=["A", "B"]).fillna(0)
    eligible = counts.index[(counts["A"] >= MIN_UNITS_PER_FIRE_SIDE) &
                            (counts["B"] >= MIN_UNITS_PER_FIRE_SIDE)]
    rates = g["mean"].unstack("side").reindex(index=eligible, columns=["A", "B"])
    details = rates.rename(columns={"A": "held_rate_A", "B": "held_rate_B"}).copy()
    details["n_units_A"] = counts.loc[eligible, "A"]
    details["n_units_B"] = counts.loc[eligible, "B"]
    details["difference_B_minus_A"] = details["held_rate_B"] - details["held_rate_A"]
    details["contrast"] = name
    delta = details["difference_B_minus_A"].to_numpy()
    lower, upper = bootstrap_mean(delta, "paired:" + name) if len(delta) else (np.nan, np.nan)
    matched = selected.loc[selected["_irwin_norm"].isin(eligible)]
    row = {
        "contrast": name, "group_A": label_a, "group_B": label_b,
        "n_paired_fires": len(eligible),
        "fire_average_held_rate_A": rates["A"].mean(),
        "fire_average_held_rate_B": rates["B"].mean(),
        "mean_difference_B_minus_A": delta.mean() if len(delta) else np.nan,
        "difference_ci_lower": lower, "difference_ci_upper": upper,
        "meets_fire_reporting_screen": len(eligible) >= MIN_GROUP_FIRES,
        "degenerate_fire_bootstrap": bool(len(delta) > 1 and np.ptp(delta) == 0),
        "evidence_status": "exploratory paired comparison; not a causal effect",
    }
    for side in ["A", "B"]:
        sub = matched.loc[matched["side"].eq(side)]
        row["n_fireline_units_" + side] = len(sub)
        row["n_source_records_" + side] = sub["source_record_key"].nunique()
        row["n_held_" + side] = int(sub["hold_label"].sum())
        row["n_burned_over_" + side] = len(sub) - int(sub["hold_label"].sum())
    return row, details.reset_index()

# Use the validated numeric bins to exclude missing/invalid predictor values.
contrast_specs = [
    ("Forecast humidity", df["bin_humidity"].eq("0–<20"),
     df["bin_humidity"].isin(["60–<80", "80–100"]), "RH <20%", "RH 60–100%"),
    ("Forecast gust", df["bin_gust"].eq("0–<5"),
     df["bin_gust"].isin(["10–<15", "15–<20", "20+"]), "Gust <5 m/s", "Gust >=10 m/s"),
    ("Opposite-side crown depth", df["bin_crown"].isin(["0–<5", "5–<10"]),
     df["bin_crown"].isin(["20–<30", "30+"]), "Crown depth <10 m", "Crown depth >=20 m"),
    ("Reported width (screened)", df["bin_width_screened"].eq(">0–<3"),
     df["bin_width_screened"].isin(["6–<12", "12–<30", "30+"]),
     "Reported width <3 m", "Reported width >=6 m"),
]
contrast_rows, contrast_details = [], []
for name, a, b, label_a, label_b in contrast_specs:
    row, details = paired_fire_contrast(df, a, b, name, label_a, label_b)
    contrast_rows.append(row)
    contrast_details.append(details)
tables["within_fire_paired_contrasts"] = pd.DataFrame(contrast_rows)
tables["within_fire_paired_contrast_details"] = pd.concat(contrast_details, ignore_index=True)
display(tables["within_fire_paired_contrasts"])


## 6. Link to the existing eight condition groups (05I)

If the saved 05I files are available, recover the P1–P8 memberships and verify the original counts and observed Held fractions. This is a continuity check, not a new rule-learning experiment.

The original surrogate tree approximated LightGBM predictions. It did **not** directly learn rules from Held/Burned Over labels. Its final rules used all study years, including median-imputed predictor values. Therefore, outcome summaries and year slices of P1–P8 are exploratory. A group below the study-average Held fraction is not automatically a majority-Burned-Over group.


In [ ]:
legacy_status = "Skipped: saved 05I files unavailable."
if LEGACY_ASSIGNMENT_PATH.is_file() and LEGACY_RULE_PATH.is_file():
    legacy = read_columns(LEGACY_ASSIGNMENT_PATH,
                          ["unit_uid", "_irwin_norm", "Year", "hold_label", "physical_only_regime_id"])
    check_unique(legacy, ["unit_uid"], "Legacy regime assignments")
    require(identity_digest(legacy) == master_identity, "Legacy cache belongs to a different population.")
    require(legacy["physical_only_regime_id"].notna().all(), "Missing legacy regime membership.")
    df = df.merge(legacy[["unit_uid", "physical_only_regime_id"]], on="unit_uid",
                  how="left", validate="one_to_one")
    rules = pd.read_csv(LEGACY_RULE_PATH)
    check_unique(rules, ["regime_id"], "Legacy rule summary")
    summaries = summarize_groups(df, ["physical_only_regime_id"], "Legacy 05I descriptive groups")
    summaries = summaries.merge(
        rules[["regime_id", "rule", "units", "incidents", "unit_held_rate", "incident_balanced_held_rate"]],
        left_on="physical_only_regime_id", right_on="regime_id", how="left",
        validate="one_to_one", indicator="_rule_match")
    require(summaries["_rule_match"].eq("both").all() and len(summaries) == 8, "Expected eight matched groups.")
    require((summaries["n_fireline_units"] == summaries["units"]).all(), "Legacy unit counts changed.")
    require((summaries["n_fires"] == summaries["incidents"]).all(), "Legacy fire counts changed.")
    require(np.allclose(summaries["pooled_held_rate"], summaries["unit_held_rate"], atol=1e-10),
            "Legacy pooled Held fractions changed.")
    require(np.allclose(summaries["fire_average_held_rate"], summaries["incident_balanced_held_rate"], atol=1e-10),
            "Legacy fire-average Held fractions changed.")
    tables["legacy_regimes_reproduced"] = summaries.drop(columns="_rule_match")
    tables["legacy_regimes_by_year_descriptive"] = summarize_groups(
        df, ["Year", "physical_only_regime_id"], "Legacy groups by year: NOT forward validation", False)
    legacy_status = "Verified eight saved 05I groups; counts and observed rates reproduced."
    display(tables["legacy_regimes_reproduced"][[
        "physical_only_regime_id", "n_fireline_units", "n_fires",
        "pooled_held_rate", "fire_average_held_rate", "held_ci_lower", "held_ci_upper"]])
print(legacy_status)


## 7. Figures for inspecting the evidence

Blue marks meet the configured count screen; grey marks do not, or have a degenerate bootstrap. The screen is not an assessment of causal validity. All groups remain in the exported tables. Probability axes run from 0 to 1, and the combination heatmap uses one common scale.

These are analysis figures, not automatically approved paper results. Inspect coverage, uncertainty, and agreement across fires before choosing what belongs in the manuscript.


In [ ]:
figure_paths = []
def save_figure(fig, stem):
    for extension in ["png", "pdf"]:
        path = FIGURE_DIR / f"{stem}.{extension}"
        fig.savefig(path, bbox_inches="tight", facecolor="white")
        figure_paths.append(path)
    plt.show()
    plt.close(fig)

width_order = ["Missing", "Zero", "Negative", "Non-numeric", "Nonfinite",
               "Positive, at or below review flag", "Positive, above review flag"]
coverage = tables["attribute_width_coverage"]
fig, ax = plt.subplots(figsize=(10.5, 4.5))
populations = ["Distinct source records represented in final dataset", "Final fireline units"]
y = np.arange(len(width_order))
for offset, population, color, label in [
    (-0.18, populations[0], "#0072B2", "Distinct source records"),
    (0.18, populations[1], "#D55E00", "Fireline units")]:
    part = coverage.loc[coverage["population"].eq(population)].set_index("width_status")
    values = part["record_fraction"].reindex(width_order).fillna(0)
    ax.barh(y+offset, values, height=0.34, color=color, label=label)
ax.set(yticks=y, yticklabels=width_order, xlim=(0, 1),
       xlabel="Fraction of the indicated population",
       title="Reported width availability in the retained dataset")
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.invert_yaxis()
ax.legend(loc="lower right")
fig.text(0.02, -0.025, f"Review flag: {WIDTH_REVIEW_THRESHOLD_FT:g} ft. Positive values are not independently verified widths.",
         fontsize=9)
save_figure(fig, "08A_width_coverage")

profiles = tables["condition_profiles"]
profile_panels = [("humidity", "Forecast relative humidity, 0–18 h (%)"),
                  ("gust", "Forecast gust at +18 h (m/s)"),
                  ("crown", "Opposite-side crown depth (m)")]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, (key, title) in zip(axes, profile_panels):
    part = profiles.loc[profiles["predictor_key"].eq(key)].set_index("condition_group")
    part = part.reindex(SPECS[key]["labels"])
    for position, label in enumerate(SPECS[key]["labels"]):
        row = part.loc[label]
        if pd.isna(row["fire_average_held_rate"]):
            ax.text(position, 0.04, "No data", ha="center", rotation=90, fontsize=8)
            continue
        adequate = bool(row["meets_reporting_screen"]) and not bool(row["degenerate_fire_bootstrap"])
        color = "#0072B2" if adequate else "#888888"
        mean = row["fire_average_held_rate"]
        low, high = row["held_ci_lower"], row["held_ci_upper"]
        if np.isfinite(low) and np.isfinite(high):
            ax.vlines(position, low, high, color=color, linewidth=1.8)
            ax.hlines([low, high], position-0.07, position+0.07, color=color)
        ax.plot(position, mean, "o", color=color)
        ax.text(position, 0.025, f"{int(row['n_fires'])} fires", ha="center", va="bottom",
                rotation=90, fontsize=8)
    ax.set(xticks=range(len(part)), xticklabels=SPECS[key]["labels"], ylim=(0, 1),
           title=title, xlabel="Condition group")
    ax.tick_params(axis="x", labelrotation=35)
    ax.yaxis.set_major_formatter(PercentFormatter(1))
axes[0].set_ylabel("Fire-average recorded Held fraction")
fig.suptitle("Observed outcomes by condition; bars are fire-bootstrap intervals", y=1.02)
fig.tight_layout()
save_figure(fig, "08A_condition_profiles")

joint = tables["condition_combinations"].query("first_predictor == 'humidity' and second_predictor == 'gust'")
row_order, column_order = SPECS["humidity"]["labels"], SPECS["gust"]["labels"]
values = np.full((len(row_order), len(column_order)), np.nan)
counts = {}
for i, rh in enumerate(row_order):
    for j, gust in enumerate(column_order):
        cell = joint.loc[joint["first_group"].eq(rh) & joint["second_group"].eq(gust)]
        if len(cell):
            row = cell.iloc[0]
            counts[(i, j)] = row
            if bool(row["meets_reporting_screen"]) and not bool(row["degenerate_fire_bootstrap"]):
                values[i, j] = row["fire_average_held_rate"]
fig, ax = plt.subplots(figsize=(8, 5.5))
cmap = plt.get_cmap("cividis").copy()
cmap.set_bad("#e6e6e6")
im = ax.imshow(np.ma.masked_invalid(values), cmap=cmap, vmin=0, vmax=1, aspect="auto")
for i in range(len(row_order)):
    for j in range(len(column_order)):
        row = counts.get((i, j))
        if row is None:
            label = "No data"
        elif np.isnan(values[i, j]):
            label = f"Limited\n{int(row['n_fires'])} fires"
        else:
            label = f"{values[i, j]:.0%}\n{int(row['n_fires'])} fires"
        color = "white" if np.isfinite(values[i, j]) and values[i, j] < 0.45 else "#111111"
        ax.text(j, i, label, ha="center", va="center", fontsize=9, color=color)
ax.set(xticks=range(len(column_order)), xticklabels=column_order,
       yticks=range(len(row_order)), yticklabels=row_order,
       xlabel="Forecast gust at +18 h (m/s)", ylabel="Mean forecast relative humidity, 0–18 h (%)",
       title="Recorded Held outcomes under combined weather conditions")
fig.colorbar(im, ax=ax, label="Fire-average recorded Held fraction", format=PercentFormatter(1))
fig.text(0.02, -0.035, "Grey cells have no data or fail the reporting screen. Missing values and all counts remain in the CSV.",
         fontsize=9)
save_figure(fig, "08A_humidity_gust_combinations")

contrasts = tables["within_fire_paired_contrasts"]
fig, ax = plt.subplots(figsize=(10, 4.8))
for i, row in contrasts.iterrows():
    if not np.isfinite(row["mean_difference_B_minus_A"]):
        ax.text(0, i, "No eligible paired fires", va="center", fontsize=9)
        continue
    color = "#0072B2" if row["meets_fire_reporting_screen"] and not row["degenerate_fire_bootstrap"] else "#888888"
    ax.plot(100*row["mean_difference_B_minus_A"], i, "o", color=color)
    if np.isfinite(row["difference_ci_lower"]) and np.isfinite(row["difference_ci_upper"]):
        ax.hlines(i, 100*row["difference_ci_lower"], 100*row["difference_ci_upper"], color=color, linewidth=2)
ax.axvline(0, color="#444444", linestyle="--", linewidth=1)
labels = [f"{r.group_B} minus {r.group_A}\n{r.n_paired_fires} paired fires"
          for r in contrasts.itertuples()]
ax.set(yticks=range(len(contrasts)), yticklabels=labels,
       xlabel="Difference in recorded Held fraction (percentage points)",
       title="Condition differences within the same fires")
ax.invert_yaxis()
fig.text(0.02, -0.035, "Each fire receives equal weight. These comparisons do not control all differences within a fire.",
         fontsize=9)
save_figure(fig, "08A_within_fire_contrasts")


## 8. Save an auditable run and a concise findings report

All tables and the enriched selected-column analysis table are saved in this run's new directory. The table is not a replacement for the 803-predictor master; it carries the selected predictors, original prediction timestamps, and recovered attributes used here. The report separates evidence already present from work still required. It does not automatically promote a condition group to a prediction rule.

The original master and source files are never written. Final checks confirm the same fireline units, outcomes, and selected predictor values, and unchanged input-file sizes/timestamps.


In [ ]:
require(identity_digest(df) == master_identity and len(df) == EXPECTED_UNITS, "Final population changed.")
final_master_view = df[master.columns].sort_values("unit_uid")
final_values_hash = hashlib.sha256(
    pd.util.hash_pandas_object(final_master_view, index=False).to_numpy().tobytes()).hexdigest()
require(final_values_hash == master_values_hash, "An original selected predictor or identity value changed.")
for path in input_paths:
    old = input_stats[str(path)]
    require(path.stat().st_size == old["bytes"] and path.stat().st_mtime_ns == old["modified_ns"],
            "An input changed while the notebook was running: " + str(path))

for name, table in tables.items():
    table.to_csv(RUN_DIR / (name + ".csv"), index=False, encoding="utf-8-sig")
df.to_parquet(RUN_DIR / "08A_enriched_frozen_fireline_units.parquet", index=False)

manifest = {
    "build_id": BUILD_ID, "run_id": RUN_ID, "is_smoke_test": IS_SMOKE_TEST,
    "completed_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": PROJECT_ROOT, "run_directory": RUN_DIR,
    "python": sys.version, "platform": platform.platform(),
    "package_versions": {"numpy": np.__version__, "pandas": pd.__version__,
                         "pyarrow": __import__("pyarrow").__version__,
                         "matplotlib": __import__("matplotlib").__version__},
    "seed": RANDOM_SEED, "bootstrap_replicates": BOOTSTRAP_REPLICATES,
    "minimum_group_units": MIN_GROUP_UNITS, "minimum_group_fires": MIN_GROUP_FIRES,
    "minimum_outcome_count": MIN_OUTCOME_COUNT, "minimum_units_per_fire_per_contrast_side": MIN_UNITS_PER_FIRE_SIDE,
    "width_review_flag_ft": WIDTH_REVIEW_THRESHOLD_FT,
    "width_review_sensitivity_ft": WIDTH_REVIEW_SENSITIVITY_FT,
    "input_files": input_stats,
    "identity_hash": master_identity, "selected_master_values_hash": master_values_hash,
    "prediction_time_columns_preserved": anchor_columns,
    "represented_source_attribute_hash": parent_attribute_hash,
    "notebook_source_hash": notebook_source_hash,
    "n_fireline_units": len(df), "n_fires": df["_irwin_norm"].nunique(),
    "n_distinct_source_records": df["source_record_key"].nunique(),
    "numeric_group_definitions": SPECS, "pair_definitions": PAIR_SPECS,
    "contrasts": [{"name": name, "group_A": a, "group_B": b}
                  for name, _, _, a, b in contrast_specs],
    "legacy_check": legacy_status,
    "interpretation": "Exploratory fixed-group recorded-outcome analysis; no training, causal claim, or new independent validation.",
}
write_json("08A_run_manifest.json", manifest)
write_json("08A_quality_checks.json", {
    "all_source_records_matched": True, "source_outcomes_agree": True,
    "master_population_preserved": True, "selected_master_values_preserved": True,
    "prediction_timestamps_preserved": True,
    "saved_fire_folds_preserved": True, "input_stats_unchanged": True,
    "no_width_imputation": True, "is_smoke_test": IS_SMOKE_TEST,
    "legacy_check": legacy_status,
})

n_screened = int(df["reported_width_m_screened"].notna().sum())
n_screened_parents = parents.loc[parents["reported_width_m_screened"].notna(), "source_record_key"].nunique()
n_screened_fires = df.loc[df["reported_width_m_screened"].notna(), "_irwin_norm"].nunique()
outcome = tables["study_outcome_summary"].iloc[0]
def report_row(values):
    return "| " + " | ".join(str(x).replace("|", "/").replace("\n", " ") for x in values) + " |"
lines = [
    "# 08A — Fireline characteristics and condition–outcome analysis",
    "",
    "**SMOKE TEST — reduced bootstrap; not a paper result.**" if IS_SMOKE_TEST else "**Exploratory analysis, not an operational rule.**",
    "",
    f"- Run: {RUN_ID}; bootstrap replicates: {BOOTSTRAP_REPLICATES:,}.",
    f"- Frozen population: {len(df):,} fireline units, {df['_irwin_norm'].nunique():,} fires, {len(parents):,} distinct source records.",
    f"- Pooled Held fraction: {outcome['pooled_held_rate']:.3f}; fire-average Held fraction: {outcome['fire_average_held_rate']:.3f}.",
    "- The two fractions answer different questions: average fireline unit versus equally weighted fire.",
    "",
    "## Attribute readiness",
    f"Positive width at or below the {WIDTH_REVIEW_THRESHOLD_FT:g} ft review flag is available for "
    f"{n_screened:,} fireline units ({n_screened/len(df):.1%}), "
    f"{n_screened_parents:,} distinct source records, and {n_screened_fires:,} fires.",
    "These are reported widths, not independently checked measurements. Repeated child fireline units do not provide independent width measurements.",
    "Review attribute_width_coverage.csv, width_coverage_per_fire.csv, and width_review_flag_sensitivity.csv before deciding whether width merits a paper result.",
    "Source-record length is not total constructed fireline length; analytical fireline-unit length is capped by design.",
    "Forecast relative humidity is not fuel moisture. No measured fuel-moisture result is produced.",
    "",
    "## Condition profiles",
    "condition_profiles.csv and condition_combinations.csv contain observed Held and Burned Over fractions, fire/source-record/unit counts, uncertainty, and support flags.",
    "Do not interpret the largest descriptive fraction as a validated rule. The groups may contain different fires and environmental conditions.",
    "Year and fold slices are descriptive; they are not new out-of-sample tests.",
    "",
    "## Matched within-fire comparisons",
    "Positive differences mean a higher recorded Held fraction in group B than A. Each row uses the same fires on both sides.",
    report_row(["Comparison (B minus A)", "Paired fires", "Difference (percentage points)", "95% fire-bootstrap interval", "Support screen"]),
    "|---|---:|---:|---:|---|",
]
for row in tables["within_fire_paired_contrasts"].itertuples():
    value = f"{100*row.mean_difference_B_minus_A:.1f}" if np.isfinite(row.mean_difference_B_minus_A) else "Not estimable"
    interval = (f"{100*row.difference_ci_lower:.1f} to {100*row.difference_ci_upper:.1f}"
                if np.isfinite(row.difference_ci_lower) else "Not estimable")
    support = ("Meets fire-count screen" if row.meets_fire_reporting_screen else "Few paired fires")
    if row.degenerate_fire_bootstrap:
        support += "; degenerate bootstrap"
    lines.append(report_row([f"{row.group_B} minus {row.group_A}", row.n_paired_fires, value, interval, support]))
lines += [
    "",
    "Intervals resample observed fires with fixed groups. They do not account for exploratory group choice, selection bias, or retraining.",
    "Within-fire comparisons do not control all differences in placement, vegetation, suppression activity, or timing.",
    "",
    "## Existing regime check", legacy_status,
    "Saved P1–P8 rules were constructed using all years to approximate model probabilities. Their observed outcomes are descriptive, not a new validation.",
    "",
    "## Next: 08B (not run or created by 08A)",
    "Use 08A to assess which attributes are interpretable and sufficiently covered. Develop a small set of rules without using evaluation-fire outcomes to choose variables or cut points.",
    "Evaluate frozen rules on excluded fires and forward years, report both outcomes and coverage, and compare against appropriate training-only references.",
    "Because this archive has already informed development, call additional internal comparisons exploratory and seek new fires for independent confirmation.",
    "",
    "## Scope",
    "The retained population was selected using recorded NIFC labels and the complete NIROPS record. Associations do not prove that constructing or widening a fireline changes fire spread.",
    "Original labels, model files, prediction times, and membership were not changed; no model was fitted.",
]
(RUN_DIR / "08A_FINDINGS_REPORT.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
write_json("08A_output_inventory.json", [
    {"path": str(p.relative_to(RUN_DIR)), "bytes": p.stat().st_size}
    for p in sorted(RUN_DIR.rglob("*")) if p.is_file()
])
display(Markdown("\n".join(lines)))
print("\nSaved run:", RUN_DIR)
print("Share 08A_FINDINGS_REPORT.md, attribute_width_coverage.csv, and within_fire_paired_contrasts.csv first.")


## What to do after the run

1. Review the width coverage and extreme-value flags before making claims about construction dimensions.
2. Inspect the condition-profile and paired-comparison tables together. Note groups supported by few fires, disagreement between pooled and fire-average summaries, and differences across years.
3. Share **08A_FINDINGS_REPORT.md** and the main tables for interpretation. The notebook does not choose a favorable subset automatically.
4. Use a separate **08B** notebook to develop and evaluate interpretable rules. Do not edit the existing 05/06 models or manuscript conclusions solely because one exploratory group has a high Held fraction.

A defensible eventual paper statement would report a condition group's **recorded outcome fraction, number of fires and fireline units, uncertainty, and independent test performance where available**. It would not claim that the conditions guarantee Held or that changing fireline width causes the outcome.
